In [1]:
import pandas as pd
import io
import zipfile
from google.colab import files

print("Vui lòng bấm 'Choose Files' để tải file dataset.zip lên:")
uploaded = files.upload()

# Lấy tên file zip vừa upload
filename = list(uploaded.keys())[0]

# Mở file zip và đọc file .csv bên trong mà không cần giải nén ra ổ cứng
with zipfile.ZipFile(io.BytesIO(uploaded[filename]), 'r') as z:
    # Tìm tên file .csv có bên trong file zip
    csv_filename = [f for f in z.namelist() if f.endswith('.csv')][0]

    # Đọc file .csv đó bằng pandas
    with z.open(csv_filename) as f:
        df = pd.read_csv(f)

print(f"\nĐã tải và giải nén thành công file chứa trong '{filename}'!")
print("Xem trước 3 dòng đầu tiên:")
display(df.head(3))

Vui lòng bấm 'Choose Files' để tải file dataset.zip lên:


Saving archive.zip to archive.zip

Đã tải và giải nén thành công file chứa trong 'archive.zip'!
Xem trước 3 dòng đầu tiên:


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1


In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. Chuẩn bị lại dữ liệu (làm sạch số 0 giống bước EDA)
cols_to_fix = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_to_fix] = df[cols_to_fix].replace(0, np.nan)
for col in cols_to_fix:
    df[col] = df[col].fillna(df[col].median())

# Chia tập Features (X) và Target (y)
X = df.drop(columns=['Outcome', 'index'] if 'index' in df.columns else ['Outcome'])
y = df['Outcome']

# Chia tập Train / Test (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. Chuẩn hóa dữ liệu (Rất quan trọng cho Logistic Regression và SVM)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3. Khởi tạo 4 mô hình Machine Learning
models = {
    "Logistic Regression": LogisticRegression(random_state=42),
    "SVM": SVC(probability=True, random_state=42),
    "Naive Bayes": GaussianNB(),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

print("="*50)
print("--- KẾT QUẢ ĐÁNH GIÁ 4 MÔ HÌNH TRÊN TẬP TEST ---")
print("="*50)

for name, model in models.items():
    # Chọn dữ liệu: SVM và Logistic dùng dữ liệu đã Scale
    if name in ["Logistic Regression", "SVM"]:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)

    print(f"\n> Mô hình: {name}")
    print(f"  - Độ chính xác (Accuracy): {acc * 100:.2f}%")
    print("  - Báo cáo chi tiết (Classification Report):")
    print(classification_report(y_test, y_pred))
    print("-" * 50)

--- KẾT QUẢ ĐÁNH GIÁ 4 MÔ HÌNH TRÊN TẬP TEST ---

> Mô hình: Logistic Regression
  - Độ chính xác (Accuracy): 70.78%
  - Báo cáo chi tiết (Classification Report):
              precision    recall  f1-score   support

           0       0.75      0.82      0.78       100
           1       0.60      0.50      0.55        54

    accuracy                           0.71       154
   macro avg       0.68      0.66      0.67       154
weighted avg       0.70      0.71      0.70       154

--------------------------------------------------

> Mô hình: SVM
  - Độ chính xác (Accuracy): 74.03%
  - Báo cáo chi tiết (Classification Report):
              precision    recall  f1-score   support

           0       0.78      0.84      0.81       100
           1       0.65      0.56      0.60        54

    accuracy                           0.74       154
   macro avg       0.71      0.70      0.70       154
weighted avg       0.73      0.74      0.73       154

----------------------------------

In [ ]:
import pandas as pd
import numpy as np
import time  # Thư viện để đo thời gian
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier

# 1. Khởi tạo từ điển chứa 4 mô hình
models = {
    "Logistic Regression": LogisticRegression(random_state=42),
    "SVM": SVC(probability=True, random_state=42),
    "Naive Bayes": GaussianNB(),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

results_data = []

# 2. Vòng lặp huấn luyện, dự đoán và đo thời gian
for name, model in models.items():
    # Chọn dữ liệu phù hợp (SVM và Logistic dùng dữ liệu đã Scale)
    if name in ["Logistic Regression", "SVM"]:
        model.fit(X_train_scaled, y_train)

        # Bắt đầu bấm giờ dự đoán
        start_time = time.time()
        y_pred = model.predict(X_test_scaled)
        end_time = time.time()

        y_prob = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)

        # Bắt đầu bấm giờ dự đoán
        start_time = time.time()
        y_pred = model.predict(X_test)
        end_time = time.time()

        y_prob = model.predict_proba(X_test)[:, 1]

    # Tính thời gian dự đoán (đổi ra mili-giây (ms) để số hiển thị dễ nhìn hơn)
    pred_time_ms = (end_time - start_time) * 1000

    # Tính các chỉ số đánh giá
    acc = accuracy_score(y_test, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='weighted')
    roc_auc = roc_auc_score(y_test, y_prob)

    # Lưu kết quả
    results_data.append({
        "Mô hình": name,
        "Accuracy (%)": round(acc * 100, 2),
        "Precision (%)": round(precision * 100, 2),
        "Recall (%)": round(recall * 100, 2),
        "F1-Score (%)": round(f1 * 100, 2),
        "ROC-AUC": round(roc_auc, 4),
        "Thời gian dự đoán (ms)": round(pred_time_ms, 2)
    })

# 3. Tạo DataFrame tổng hợp và hiển thị bảng
df_summary = pd.DataFrame(results_data)

print("="*95)
print("BẢNG TỔNG HỢP SO SÁNH HIỆU SUẤT VÀ TỐC ĐỘ CÁC MÔ HÌNH")
print("="*95)
display(df_summary)

# Xuất dạng Markdown để bạn dán thẳng vào báo cáo nếu cần
#print("\n--- Định dạng Markdown cho báo cáo ---")
#print(df_summary.to_markdown(index=False))

BẢNG TỔNG HỢP SO SÁNH HIỆU SUẤT VÀ TỐC ĐỘ CÁC MÔ HÌNH


,Mô hình,Accuracy (%),Precision (%),Recall (%),F1-Score (%),ROC-AUC,Thời gian dự đoán (ms)
0,Logistic Regression,70.78,69.89,70.78,70.08,0.8130,0.55
1,SVM,74.03,73.37,74.03,73.49,0.7964,5.61
2,Naive Bayes,70.13,70.99,70.13,70.45,0.7646,1.41
3,Random Forest,77.92,77.45,77.92,77.32,0.8192,18.53
